# Patient-conditioned ECG forecasting

Run the script workflow in README first. Select the project virtual-environment kernel. This notebook reads measured results; it does not invent metrics or retrain models when opened.

In [ ]:
from pathlib import Path
import os
if not Path('src').exists() and Path('../src').exists():
    os.chdir('..')
import sys
sys.path.insert(0, str(Path('src').resolve()))
from config import load_config, read_json
cfg = load_config('configs/pilot.yaml')
manifest = read_json('data/pilot/manifest.json')
manifest['protocol']

## Patient splits and conditioning

Age and sex are fixed covariates. NORM/MI/STTC/CD/HYP history comes only from strictly earlier ECGs. Current-record diagnoses below are used only for stratified evaluation.

In [ ]:
import pandas as pd
meta = {s: pd.read_csv(f'data/pilot/metadata_{s}.csv') for s in ('train','val','test')}
pd.DataFrame({s: {'records': len(m), 'patients': m.patient_id.nunique(), 'with_history': int(m.history_available.sum())} for s,m in meta.items()}).T

## Model input/output shapes

500 Hz, 300 past samples (0.6 s), 150 future samples (0.3 s), all 12 leads. Contrastive projection receives observed dynamics before static fusion.

In [ ]:
from preprocessing import ECGDataset
from models import build_model
ds = ECGDataset(cfg, 'train', 'patient_history')
batch = ds[0]
model = build_model(cfg, 'cnn_lstm', 10).eval()
import torch
with torch.inference_mode():
    forecast = model(batch['x'][None], batch['static'][None])
print(batch['x'].shape, batch['static'].shape, forecast.shape)

In [ ]:
pd.read_csv('outputs/pilot/model_comparison.csv')

In [ ]:
pd.read_csv('outputs/pilot/paired_patient_bootstrap.csv')

In [ ]:
from IPython.display import Image, display
display(Image('outputs/pilot/model_comparison.png'))
display(Image('outputs/pilot/cnn_lstm_patient_history/learning_curves.png'))
display(Image('outputs/pilot/cnn_lstm_patient_history/forecast_lowest_id.png'))

## Discuss

Compare demographic versus ECG-only, history versus demographics, and contrastive versus history-only runs. Negative paired B-A intervals favor B. Compare architectures under each condition. Inspect sharp peaks and timing as well as mean error. Read `Project_Documents/ASSIGNMENT_REPORT.md` and `Project_Documents/PRESENTATION_NOTES.md` for equations and literature caveats.